In [1]:
# ── LIX GPU Server Setup ─────────────────────────────────────────────────────
# No Google Drive here. Your dataset lives in a normal folder on the server.
# Set DATASET_DIR to wherever you copied the VQA-RAD_dataset folder.
# (Default assumes you put it in your home directory: /home/matei/VQA-RAD_dataset)

import os

DATASET_DIR = "/home/matei/VQA-RAD_dataset"

assert os.path.isdir(DATASET_DIR), (
    f"Dataset folder not found at {DATASET_DIR}. "
    "Copy your VQA-RAD_dataset folder to the server first (see instructions)."
)
print(f"Dataset folder found: {DATASET_DIR}")


Dataset folder found: /home/matei/VQA-RAD_dataset


In [2]:
import json
import os
from PIL import Image
from datasets import Dataset, DatasetDict


dataset_dir = DATASET_DIR
image_dir   = os.path.join(dataset_dir, "VQA_RAD Image Folder")

with open(os.path.join(dataset_dir, "VQA_RAD Dataset Public.json")) as f:
    records = json.load(f)

def load_records(recs, add_framed):
    rows = {"image_path": [], "question": [], "answer": [], "question_type": [], "answer_type": [], "image_organ": []}
    seen = set()

    def add_row(img_path, question, rec):
        # skip exact-duplicate (image, question, answer) rows
        key = (img_path, " ".join(str(question).strip().lower().split()), str(rec["answer"]).strip().lower())
        if key in seen:
            return
        seen.add(key)
        rows["image_path"].append(img_path)
        rows["question"].append(str(question))
        rows["answer"].append(str(rec["answer"]))
        rows["question_type"].append(str(rec.get("question_type", "OTHER")))
        rows["answer_type"].append(str(rec.get("answer_type", "OTHER")).strip().upper())
        rows["image_organ"].append(str(rec.get("image_organ", "")))

    for rec in recs:
        img_path = os.path.join(image_dir, rec["image_name"])
        if not os.path.isfile(img_path):
            continue

        add_row(img_path, rec["question"], rec)              # the record's own question (free-form or paraphrase)

        # Paraphrase ("rephrase") questions are ALREADY separate records, so re-adding the
        # question_rephrase field only duplicates them -> we do NOT. The templated "framed"
        # questions exist ONLY in this field (never as records), so we add them, but to TRAIN
        # only: adding a test question's framing to train would leak the test set.
        if add_framed:
            frame = rec.get("question_frame", "NULL")
            if frame and frame != "NULL":
                add_row(img_path, frame, rec)

    return Dataset.from_dict(rows)

dataset = DatasetDict({
    "train": load_records([r for r in records if not r["phrase_type"].startswith("test_")], add_framed=True),
    "test":  load_records([r for r in records if     r["phrase_type"].startswith("test_")], add_framed=False),
})

first_row = dataset["train"][0]

print(f"Dataset loaded successfully!")
print(f"  Train : {len(dataset['train'])} examples")
print(f"  Test  : {len(dataset['test'])}  examples")
print(f"Sample entry:")
print(f"  Question : {first_row['question']}")
print(f"  Answer   : {first_row['answer']}")
print(f"  Image    : {first_row['image_path']}")


/home/matei/miniconda3/envs/vlm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset loaded successfully!
  Train : 2393 examples
  Test  : 451  examples
Sample entry:
  Question : Are regions of the brain infarcted?
  Answer   : Yes
  Image    : /home/matei/VQA-RAD_dataset/VQA_RAD Image Folder/synpic54610.jpg


In [3]:
from collections import Counter

print("Analyzing Training Dataset Balance...\n")

# 1. Extract all the answers from the training split
# We convert them to strings, strip whitespace, and make them lowercase 
# so that "Yes", "yes", and " yes " are all counted as the same thing.
all_answers = [str(ans).strip().lower() for ans in dataset['train']['answer']]

# 2. Count how many times each answer appears
answer_counts = Counter(all_answers)

# 3. Print the top 10 most common answers overall
print("--- Top 10 Most Common Answers ---")
for ans, count in answer_counts.most_common(10):
    print(f"'{ans}': {count} times")

# 4. Check the exact balance of Yes/No questions
yes_count = answer_counts.get('yes', 0)
no_count = answer_counts.get('no', 0)
total_binary = yes_count + no_count

print("\n--- Yes/No Question Balance ---")
if total_binary > 0:
    yes_pct = (yes_count / total_binary) * 100
    no_pct = (no_count / total_binary) * 100
    print(f"Total Yes/No Questions: {total_binary}")
    print(f"YES: {yes_count} ({yes_pct:.1f}%)")
    print(f"NO:  {no_count} ({no_pct:.1f}%)")
    
    # Give a warning if it is highly imbalanced
    if yes_pct > 70 or no_pct > 70:
        print("\n⚠️ WARNING: Your Yes/No classes are highly imbalanced!")
        print("The model might just memorize the most frequent answer instead of learning.")
else:
    print("No 'Yes' or 'No' answers found.")

Analyzing Training Dataset Balance...

--- Top 10 Most Common Answers ---
'no': 667 times
'yes': 638 times
'axial': 31 times
'right': 27 times
'left': 19 times
'pa': 14 times
'ct': 10 times
'pancreas': 10 times
'one': 10 times
'left kidney': 10 times

--- Yes/No Question Balance ---
Total Yes/No Questions: 1305
YES: 638 (48.9%)
NO:  667 (51.1%)


In [4]:
# IMPORTANT (LIX server): models cannot be auto-downloaded from HuggingFace.
# Tell the libraries to stay offline, and load from the local folder where
# download_llms.sh placed the weights. Verify the exact path with: ls /home/matei
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

# %pip install transformers accelerate

import torch
from transformers import Blip2Processor, Blip2ForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

# float16 on CUDA; bfloat16 on MPS (Apple Silicon supports bfloat16 and it halves
# peak memory vs float32, avoiding OOM on the 2.7B model); float32 on CPU
if device == "cuda":
    dtype = torch.float16
elif device == "mps":
    dtype = torch.bfloat16
else:
    dtype = torch.float32
print(f"Using dtype : {dtype}")

# Local path to the downloaded model (NOT the HuggingFace name).
# If ls /home/matei shows a different folder name, update this string.
CHECKPOINT = "/home/matei/blip2-opt-2.7b"
print("Downloading and loading BLIP-2...")

processor = Blip2Processor.from_pretrained(CHECKPOINT)

# low_cpu_mem_usage=True loads weights layer-by-layer instead of
# loading everything into RAM at once before casting — halves peak memory
model = Blip2ForConditionalGeneration.from_pretrained(
    CHECKPOINT,
    torch_dtype=dtype,
    low_cpu_mem_usage=True,
)
model = model.to(device)
model.eval()

print("BLIP-2 loaded successfully!")

# Sanity check on the first dataset row
sample_image  = Image.open(first_row["image_path"]).convert("RGB")
sample_prompt = f"Question: {first_row['question']} Answer:"

inputs = processor(
    images=sample_image,
    text=sample_prompt,
    return_tensors="pt",
).to(device)
inputs["pixel_values"] = inputs["pixel_values"].to(dtype)

with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=10)

# generated_ids = [prompt_tokens..., new_tokens...]; slice off the prompt
input_len  = inputs["input_ids"].shape[1]
prediction = processor.batch_decode(generated_ids[:, input_len:], skip_special_tokens=True)[0].strip()

print(f"Sample question : {first_row['question']}")
print(f"Ground truth    : {first_row['answer']}")
print(f"BLIP-2 predicts : {prediction}")

Using device: cuda
Using dtype : torch.float16


Loading weights: 100%|██████████| 1247/1247 [00:01<00:00, 1075.80it/s]


BLIP-2 loaded successfully!
Sample question : Are regions of the brain infarcted?
Ground truth    : Yes
BLIP-2 predicts : yes


In [5]:
from PIL import Image
import random
import string

def normalize(text):
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return ' '.join(text.split())

def exact_match(pred, gt):
    return normalize(pred) == normalize(gt)


print("Sample predictions (closed-ended):\n")
closed_rows = [r for r in dataset["test"] if str(r["answer_type"]).upper() == "CLOSED"]
for row in random.sample(closed_rows, 10):
    image  = Image.open(row["image_path"]).convert("RGB")
    prompt = f"Question: {row['question']} Answer:"
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(device)
    inputs["pixel_values"] = inputs["pixel_values"].to(dtype)
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=20)
    input_len  = inputs["input_ids"].shape[1]
    pred = processor.batch_decode(generated_ids[:, input_len:], skip_special_tokens=True)[0].strip()
    match = "✓" if exact_match(pred, row["answer"]) else "✗"
    print(f"{match} GT: '{row['answer']}'  |  Pred: '{pred}'")
    print(f"   Q: {row['question']}\n")

Sample predictions (closed-ended):

✗ GT: 'No'  |  Pred: 'Yes'
   Q: Does this patient have a pneumothorax?

✓ GT: 'No'  |  Pred: 'No'
   Q: Is the mass homogenous?

✗ GT: 'no'  |  Pred: 'Yes'
   Q: Is this a coronal section?

✗ GT: 'no'  |  Pred: 'Yes, it is an AP image'
   Q: Is this an AP image?

✓ GT: 'Yes'  |  Pred: 'Yes'
   Q: Is the patient lying supine?

✓ GT: 'no'  |  Pred: 'No'
   Q: Is the size of the ventricles abnormal?

✗ GT: 'No'  |  Pred: 'yes'
   Q: Is this a normal x ray?

✗ GT: 'No'  |  Pred: 'Yes'
   Q: Is the mass well circumscribed?

✗ GT: 'yes'  |  Pred: 'Yes, there is bowel gas'
   Q: Is there bowel gas?

✗ GT: 'No'  |  Pred: 'Yes, there is a pneumothorax present'
   Q: Is there a pneumothorax present?



In [6]:
import torch, string, re
from PIL import Image
from collections import Counter
from tqdm.auto import tqdm

# ── Shared helpers ────────────────────────────────────────────────────────────
def normalize(text):
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return ' '.join(text.split())

def exact_match(pred, gt):
    return normalize(pred) == normalize(gt)

def token_recall(pred, gt):
    """Fraction of ground-truth tokens that appear in the prediction (multiset)."""
    pred_tokens = Counter(normalize(pred).split())
    gt_tokens   = Counter(normalize(gt).split())
    if not gt_tokens:
        return 0.0
    overlap = sum((pred_tokens & gt_tokens).values())
    return overlap / sum(gt_tokens.values())

def print_results(label, closed_em, closed_n, open_em, open_rec, open_n):
    sep = "=" * 52
    print(f"\n{sep}")
    print(f"  {label}")
    print(f"{sep}")
    if closed_n:
        print(f"\nClosed-ended  ({closed_n} questions)")
        print(f"  Exact Match Accuracy : {closed_em}/{closed_n}  ({100*closed_em/closed_n:.2f}%)")
    if open_n:
        print(f"\nOpen-ended  ({open_n} questions)")
        print(f"  Exact Match Accuracy : {open_em}/{open_n}  ({100*open_em/open_n:.2f}%)")
        print(f"  Token Recall         : {open_rec/open_n:.4f}  ({100*open_rec/open_n:.2f}%)")
    total    = closed_n + open_n
    total_em = closed_em + open_em
    if total:
        print(f"\nOverall  ({total} questions)")
        print(f"  Exact Match Accuracy : {total_em}/{total}  ({100*total_em/total:.2f}%)")
    print()

# ── Run inference once and cache raw predictions ──────────────────────────────
# All three evaluation strategies below reuse this cache — no repeat inference.
def run_inference(hf_dataset, split_name):
    model.eval()
    cache = []
    for i in tqdm(range(len(hf_dataset)), desc=f"Inference [{split_name}]"):
        row    = hf_dataset[i]
        image  = Image.open(row['image_path']).convert("RGB")
        prompt = f"Question: {row['question']} Answer:"
        inputs = processor(images=image, text=prompt, return_tensors="pt").to(device)
        inputs["pixel_values"] = inputs["pixel_values"].to(dtype)
        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=20)
        input_len = inputs["input_ids"].shape[1]
        raw_pred  = processor.batch_decode(
            generated_ids[:, input_len:], skip_special_tokens=True
        )[0].strip()
        cache.append({
            "raw_pred":    raw_pred,
            "true_answer": str(row['answer']),
            "ans_type":    str(row['answer_type']).upper(),
        })
    return cache

test_cache  = run_inference(dataset["test"],  "test")
train_cache = run_inference(dataset["train"], "train")
print("\nInference complete. Run the three evaluation cells below.")

Inference [train]: 100%|██████████| 2393/2393 [05:39<00:00,  7.05it/s]


Inference complete. Run the three evaluation cells below.


In [10]:
# ── Strategy 1: Raw exact match (no truncation) ──────────────────────────────
# The full decoded prediction is compared directly to the ground truth.
# "Yes, the aorta is normal in size" vs "Yes" → WRONG.
# This is the strictest baseline: any verbosity from the model is penalised.

def evaluate_raw(cache, label):
    closed_em, closed_n       = 0, 0
    open_em, open_rec, open_n = 0, 0.0, 0
    for r in cache:
        pred = r["raw_pred"]
        gt   = r["true_answer"]
        if r["ans_type"] == "CLOSED":
            closed_n += 1
            if exact_match(pred, gt): closed_em += 1
        else:
            open_n += 1
            if exact_match(pred, gt): open_em += 1
            open_rec += token_recall(pred, gt)
    print_results(label, closed_em, closed_n, open_em, open_rec, open_n)

evaluate_raw(test_cache,  "TEST  — Strategy 1: raw exact match")
evaluate_raw(train_cache, "TRAIN — Strategy 1: raw exact match")


  TEST  — Strategy 1: raw exact match

Closed-ended  (272 questions)
  Exact Match Accuracy : 89/272  (32.72%)

Open-ended  (179 questions)
  Exact Match Accuracy : 4/179  (2.23%)
  Token Recall         : 0.1663  (16.63%)

Overall  (451 questions)
  Exact Match Accuracy : 93/451  (20.62%)


  TRAIN — Strategy 1: raw exact match

Closed-ended  (1406 questions)
  Exact Match Accuracy : 449/1406  (31.93%)

Open-ended  (987 questions)
  Exact Match Accuracy : 17/987  (1.72%)
  Token Recall         : 0.1362  (13.62%)

Overall  (2393 questions)
  Exact Match Accuracy : 466/2393  (19.47%)



In [11]:
# ── Strategy 2: First-word truncation for closed-ended ───────────────────────
# For closed-ended questions, only the first word of the prediction is compared.
# "Yes, the aorta is normal in size" → "yes" → matches "Yes" ✓
# Works well for yes/no but breaks when the answer is not the first word:
# "The plane is axial" → "the" → does not match "axial" ✗

def evaluate_firstword(cache, label):
    closed_em, closed_n       = 0, 0
    open_em, open_rec, open_n = 0, 0.0, 0
    for r in cache:
        raw_pred = r["raw_pred"]
        gt       = r["true_answer"]
        if r["ans_type"] == "CLOSED":
            words = normalize(raw_pred).split()
            pred  = words[0] if words else ""
            closed_n += 1
            if exact_match(pred, gt): closed_em += 1
        else:
            open_n += 1
            if exact_match(raw_pred, gt): open_em += 1
            open_rec += token_recall(raw_pred, gt)
    print_results(label, closed_em, closed_n, open_em, open_rec, open_n)

evaluate_firstword(test_cache,  "TEST  — Strategy 2: first-word truncation")
evaluate_firstword(train_cache, "TRAIN — Strategy 2: first-word truncation")


  TEST  — Strategy 2: first-word truncation

Closed-ended  (272 questions)
  Exact Match Accuracy : 135/272  (49.63%)

Open-ended  (179 questions)
  Exact Match Accuracy : 4/179  (2.23%)
  Token Recall         : 0.1663  (16.63%)

Overall  (451 questions)
  Exact Match Accuracy : 139/451  (30.82%)


  TRAIN — Strategy 2: first-word truncation

Closed-ended  (1406 questions)
  Exact Match Accuracy : 682/1406  (48.51%)

Open-ended  (987 questions)
  Exact Match Accuracy : 17/987  (1.72%)
  Token Recall         : 0.1362  (13.62%)

Overall  (2393 questions)
  Exact Match Accuracy : 699/2393  (29.21%)



In [ ]:
# ── Strategy 3: Hybrid — first-word for yes/no, context search for other ─────
# Yes/No: first word is safe because the model leads with "Yes, ..." or "No, ..."
#   Checking the full context would risk false positives:
#   "Yes, there is no fracture" contains "no" but the answer is "yes".
# Other closed-ended (plane, organ, modality…): the answer may appear anywhere
#   in the generated sentence, so we search the full context with a word-boundary
#   regex to avoid partial matches ("no" inside "normal", "ct" inside "activity").

YES_NO = {"yes", "no"}

def closed_match_hybrid(raw_pred, gt):
    norm_gt   = normalize(gt)
    norm_pred = normalize(raw_pred)
    if norm_gt in YES_NO:
        words = norm_pred.split()
        return words[0] == norm_gt if words else False
    pattern = r'\b' + re.escape(norm_gt) + r'\b'
    return bool(re.search(pattern, norm_pred))

def evaluate_hybrid(cache, label):
    closed_em, closed_n       = 0, 0
    open_em, open_rec, open_n = 0, 0.0, 0
    for r in cache:
        raw_pred = r["raw_pred"]
        gt       = r["true_answer"]
        if r["ans_type"] == "CLOSED":
            closed_n += 1
            if closed_match_hybrid(raw_pred, gt): closed_em += 1
        else:
            open_n += 1
            if exact_match(raw_pred, gt): open_em += 1
            open_rec += token_recall(raw_pred, gt)
    print_results(label, closed_em, closed_n, open_em, open_rec, open_n)

evaluate_hybrid(test_cache,  "TEST  — Strategy 3: hybrid")
evaluate_hybrid(train_cache, "TRAIN — Strategy 3: hybrid")


  TEST  — Strategy 3: hybrid

Closed-ended  (272 questions)
  Exact Match Accuracy : 140/272  (51.47%)

Open-ended  (179 questions)
  Exact Match Accuracy : 4/179  (2.23%)
  Token Recall         : 0.1663  (16.63%)

Overall  (451 questions)
  Exact Match Accuracy : 144/451  (31.93%)


  TRAIN — Strategy 3: hybrid

Closed-ended  (1406 questions)
  Exact Match Accuracy : 700/1406  (49.79%)

Open-ended  (987 questions)
  Exact Match Accuracy : 17/987  (1.72%)
  Token Recall         : 0.1362  (13.62%)

Overall  (2393 questions)
  Exact Match Accuracy : 717/2393  (29.96%)



: 